# Regression data: exploration and preprocessing

This notebook loads `regression_dataset_40k.csv`, checks its structure and missing values, then builds a leakage-safe preprocessing pipeline for a regression model. Run it from this project folder so the relative CSV path resolves.


In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("regression_dataset_40k.csv")
TARGET = "Target_Regression"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place this notebook in the same folder as the dataset."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


Dataset shape: 40,000 rows × 31 columns


,Reg_Feat_1,Reg_Feat_2,Reg_Feat_3,Reg_Feat_4,Reg_Feat_5,Reg_Feat_6,Reg_Feat_7,Reg_Feat_8,Reg_Feat_9,Reg_Feat_10,...,Reg_Feat_22,Reg_Feat_23,Reg_Feat_24,Cat_Feat_1,Cat_Feat_2,Cat_Feat_3,Cat_Feat_4,Cat_Feat_5,Cat_Feat_6,Target_Regression
0,NaN,-340.566002,2.673984,36.964051,158.634184,-386.793094,125.988674,54.915578,-24.039826,-364.716253,...,-131.090223,124.460280,521.296468,Cat_Alpha,South,High,Active,Class_1,Green,-435.190718
1,240.901531,-397.679110,12.877416,7.393506,40.918711,43.724403,25.464929,-232.107683,18.080353,282.061660,...,110.559482,289.784054,-261.862089,Cat_Beta,West,Med,Inactive,Class_1,Yellow,-117.413116
2,NaN,-90.458633,65.482195,30.318914,92.977979,-20.246109,-0.681512,172.852474,10.832302,302.555744,...,-186.237708,-495.919670,-1.752873,Cat_Gamma,South,Med,Inactive,Class_1,Yellow,-117.900716
3,NaN,181.548311,-26.153448,116.456042,-169.070671,-330.782410,26.494343,-481.899550,-34.230368,-79.211225,...,100.146542,89.011773,-247.955449,Cat_Gamma,West,Low,Active,Class_1,Green,-481.403076
4,NaN,254.501668,92.107585,67.579371,105.281697,-45.135005,51.571023,1774.851796,-61.642058,-67.678330,...,-147.247059,378.292809,-615.361985,Cat_Gamma,West,Low,Active,Class_2,Yellow,-80.938251


## Inspect columns, types, and missing data

The expected data has 40,000 observations, 24 numeric predictors (`Reg_Feat_1` to `Reg_Feat_24`), 6 categorical predictors (`Cat_Feat_1` to `Cat_Feat_6`), and one numeric target. `Reg_Feat_1` is missing in about half of rows, while the other numeric features have much smaller missing rates. The target should not be imputed; rows without a target would be unusable for supervised training.


In [5]:
print("Column types:")
display(df.dtypes.value_counts().rename_axis("dtype").to_frame("count"))

missing = df.isna().sum().rename("missing_count").to_frame()
missing["missing_percent"] = (missing["missing_count"] / len(df) * 100).round(2)
display(missing.sort_values("missing_count", ascending=False).head(15))

print("Target summary:")
display(df[TARGET].describe().to_frame())


Column types:


,count
dtype,
float64,25
str,6


,missing_count,missing_percent
Reg_Feat_1,20000,50.00
Reg_Feat_24,837,2.09
Reg_Feat_4,831,2.08
Reg_Feat_10,830,2.08
Reg_Feat_11,818,2.04
Reg_Feat_5,813,2.03
Reg_Feat_7,808,2.02
Reg_Feat_22,807,2.02
Reg_Feat_19,806,2.02
Reg_Feat_23,803,2.01


Target summary:


,Target_Regression
count,40000.000000
mean,-2.057755
std,239.957393
min,-882.924295
25%,-161.848567
50%,-0.648264
75%,160.431038
max,1007.663054


In [6]:
# Define predictors and target explicitly so the target cannot enter the preprocessing pipeline.
X = df.drop(columns=TARGET)
y = df[TARGET]

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"Numeric predictors ({len(numeric_features)}): {numeric_features}")
print(f"Categorical predictors ({len(categorical_features)}): {categorical_features}")
assert len(numeric_features) == 24, "Review the numeric columns: 24 were expected."
assert len(categorical_features) == 6, "Review the categorical columns: 6 were expected."
assert y.notna().all(), "Target has missing values; decide how to handle them before training."


Numeric predictors (24): ['Reg_Feat_1', 'Reg_Feat_2', 'Reg_Feat_3', 'Reg_Feat_4', 'Reg_Feat_5', 'Reg_Feat_6', 'Reg_Feat_7', 'Reg_Feat_8', 'Reg_Feat_9', 'Reg_Feat_10', 'Reg_Feat_11', 'Reg_Feat_12', 'Reg_Feat_13', 'Reg_Feat_14', 'Reg_Feat_15', 'Reg_Feat_16', 'Reg_Feat_17', 'Reg_Feat_18', 'Reg_Feat_19', 'Reg_Feat_20', 'Reg_Feat_21', 'Reg_Feat_22', 'Reg_Feat_23', 'Reg_Feat_24']
Categorical predictors (6): ['Cat_Feat_1', 'Cat_Feat_2', 'Cat_Feat_3', 'Cat_Feat_4', 'Cat_Feat_5', 'Cat_Feat_6']


/var/folders/g9/ldd8rzbd60x38qjcxvs3__4h0000gn/T/ipykernel_14544/3896945619.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


## Split first, then fit preprocessing on training data

Median imputation is robust to the numeric features' large values and outliers. A missingness indicator is added for numeric columns with missing values, so the model can learn from the unusually high missing rate in `Reg_Feat_1`. Categorical missing values, if any appear, are filled with the most frequent training value; categories are one-hot encoded, with unseen validation/test categories safely ignored.

The split happens before fitting any imputer, scaler, or encoder. This prevents validation data from influencing preprocessing statistics.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# Fit only on training predictors; transform both partitions using the fitted steps.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training predictors after preprocessing:", X_train_processed.shape)
print("Test predictors after preprocessing:", X_test_processed.shape)
print("Training target:", y_train.shape)
print("Test target:", y_test.shape)


Training predictors after preprocessing: (32000, 66)
Test predictors after preprocessing: (8000, 66)
Training target: (32000,)
Test target: (8000,)


## Use with a regression estimator

Keep the fitted `preprocessor` together with a regression estimator in a single pipeline when training and evaluating models. This makes cross-validation and future predictions apply the same transformations consistently.


In [ ]:
# Example template (uncomment after choosing a regression model):
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
#
# model = Pipeline(steps=[
#     ("preprocessing", preprocessor),
#     ("regressor", RandomForestRegressor(random_state=42)),
# ])
# model.fit(X_train, y_train)
# predictions = model.predict(X_test)
# print("MAE:", mean_absolute_error(y_test, predictions))
# print("RMSE:", mean_squared_error(y_test, predictions) ** 0.5)
# print("R²:", r2_score(y_test, predictions))
